# Checkpoint Loader
Load any saved checkpoint and generate text.

In [1]:
import os
import math
from typing import Dict, List, Tuple

import torch
import torch.nn as nn

# ---- SET THIS TO YOUR CHECKPOINT ----
CKPT_PATH = 'checkpoints/best/val_ppl=63.70.pt'
# CKPT_PATH = 'checkpoints/best/val_ppl=.pt'
# -------------------------------------

DATA_DIR = 'data'
VOCAB_PATH = os.path.join(DATA_DIR, 'vocab.txt')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('DEVICE:', DEVICE)
print('CKPT_PATH:', CKPT_PATH)

DEVICE: cuda
CKPT_PATH: checkpoints/best/val_ppl=63.70.pt


## Load vocab

In [2]:
def load_vocab(vocab_path: str) -> Tuple[List[str], Dict[str, int]]:
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    tok2id = {t: i for i, t in enumerate(vocab)}
    return vocab, tok2id

vocab, tok2id = load_vocab(VOCAB_PATH)
pad_id = tok2id.get('<pad>', 0)
bos_id = tok2id.get('<bos>')
eos_id = tok2id.get('<eos>')

print('vocab size:', len(vocab))

vocab size: 26286


## Model

In [3]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, num_layers: int, dropout: float, pad_id: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)
        out, _ = self.lstm(e)
        out = self.drop(out)
        logits = self.fc(out)
        return logits

## Load checkpoint

In [4]:
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)

hp = ckpt['hyperparams']

print('=== Checkpoint info ===')
print(f"  path:      {CKPT_PATH}")
print(f"  epoch:     {ckpt.get('epoch', '?')}")
print(f"  val_loss:  {ckpt.get('val_loss', '?')}")
print(f"  val_ppl:   {ckpt.get('val_ppl', '?')}")
if 'train_loss' in ckpt:
    print(f"  train_loss:{ckpt['train_loss']}")
if 'global_step' in ckpt:
    print(f"  step:      {ckpt['global_step']}")
print()
print('=== Hyperparams ===')
for k, v in hp.items():
    print(f"  {k}: {v}")

# Build model from stored hyperparams
model = LSTMLM(
    vocab_size=hp['vocab_size'],
    emb_dim=hp['emb_dim'],
    hidden=hp['hidden_size'],
    num_layers=hp['num_layers'],
    dropout=hp['dropout'],
    pad_id=pad_id,
).to(DEVICE)

model.load_state_dict(ckpt['model_state_dict'])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded: {total_params:,} params")

=== Checkpoint info ===
  path:      checkpoints/best/val_ppl=63.70.pt
  epoch:     2
  val_loss:  4.154173522839277
  val_ppl:   63.69929677251757
  train_loss:3.6969139649285583

=== Hyperparams ===
  seed: 42
  seq_len: 50
  stride: 10
  batch_size: 256
  grad_clip: 1.0
  emb_dim: 256
  num_layers: 2
  vocab_size: 26286
  lr: 0.0015
  hidden_size: 2048
  dropout: 0.5
  steps_per_epoch: 1500
  max_epochs: 8
  early_stop_patience: 3
  train_split: 0.9

Model loaded: 113,050,798 params


## Sampling utilities

In [5]:
def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    return [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def generate(prompt: str, max_new_tokens: int = 120, temperature: float = 0.8, top_k: int = 50):
    ids = encode_prompt(prompt)
    out = sample_ids(model, ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    text = decode(out)
    print(f'Prompt: {prompt}')
    print(f'---')
    print(text)
    print()

print('Ready. Use generate("your prompt here") to sample text.')

Ready. Use generate("your prompt here") to sample text.


## Generate

In [6]:
generate('once upon a time')

Prompt: once upon a time
---
once upon a time, if it were not for a time to see it anywhere, he could not tell where it was that his wife was doing so. at this moment the old woman had arrived at the door for her. the girl, however, went to the cow, and said, " mother, bake your axe, give me a milk. " so they both went into the garden, and gave an apple-tree which she took from the kitchen-mother's horns, and left the cow in the milk-house; and she told him she could not take her place at his house. so she took her seat



In [7]:
generate('the princess')

Prompt: the princess
---
the princess, and saw the genie on the throne, and they said, ' what is the matter with you? it is the king of persia and you who are you, who have my wife, and you have been here; but i think i have seen myself in the greatest strait that ever ever was seen. tell me, who you have seen so much, and so yourself do not go out, and bring you to the king, who has been treated with your majesty's life? ' ' well, madam, ' said the fairy, ' do i not hear you tell me the truth of this strange princess



In [8]:
generate('in a dark forest')

Prompt: in a dark forest
---
in a dark forest, but she had no power to eat, and the bread and milk were not left to eat. then, after eating the lemon and nuts, and the women of the wood was just a long time about the wedding, and there was the festival of all kinds of flowers, but none of them had, or any thing else could be. when they came to the kitchen, he found a big wooden chest, which he had not so often described, and, as it was told in its original, she was not merely fond of caressing his friends, and he could not live to be a woman in

